# ROBERT Score Diagnosis (V1)

This notebook reads an extracted `run_context.json` and generates deterministic, auditable diagnosis outputs.

It does not call any external service and does not modify ROBERT core code.

Outputs written next to `run_context.json`:
- `diagnosis.json` (machine-readable flags and computed components)
- `diagnosis_summary.md` (short human-readable summary)

## Cell 2 Guide: Select a Run Context

Set `RUN_CONTEXT_PATH` to a specific file, or leave it as `None` to auto-select the newest `run_context.json` inside `agent/run_archive/`.

In [29]:
from pathlib import Path
from datetime import datetime, timezone
import json

RUN_CONTEXT_PATH = None

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "AGENTS.md").exists() and (candidate / "robert").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not infer project root from notebook working directory.")

PROJECT_ROOT = resolve_project_root()
RUNS_ROOT = PROJECT_ROOT / "agent" / "run_archive"

if RUN_CONTEXT_PATH is None:
    candidates = sorted(RUNS_ROOT.glob("**/run_context.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError("No run_context.json found. Run extract_context.ipynb first.")
    RUN_CONTEXT_PATH = candidates[0]
else:
    RUN_CONTEXT_PATH = Path(RUN_CONTEXT_PATH).resolve()

if not RUN_CONTEXT_PATH.exists():
    raise FileNotFoundError(f"run_context.json not found: {RUN_CONTEXT_PATH}")

context = json.loads(RUN_CONTEXT_PATH.read_text(encoding="utf-8"))
RUN_FOLDER = RUN_CONTEXT_PATH.parent

print(f"Loaded context: {RUN_CONTEXT_PATH}")
print(f"Prediction type: {context.get('pred_type')}")
print(f"Model: {context.get('ml_model')}")

Loaded context: /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260516_150946__TOF_class/run_context.json
Prediction type: clas
Model: RF


## Cell 4 Guide: Helper Functions

These are lightweight helpers for normalizing ROBERT-reported text fields and recording structured observations.

No ROBERT score is recomputed here.

In [30]:
def parse_ratio(text):
    """Split a ROBERT-reported ratio string like '106:3' into numeric values."""
    if not text or ":" not in str(text):
        return None, None
    left, right = str(text).split(":", 1)
    try:
        return float(left), float(right)
    except ValueError:
        return None, None


def normalize_verify_results(test_results):
    """Normalize VERIFY results into a stable list of {'test': ..., 'result': ...}.

    Supports list format like ['y_mean: PASSED', ...] and dict format when available.
    """
    normalized = []
    if isinstance(test_results, list):
        for item in test_results:
            if not isinstance(item, str) or ":" not in item:
                continue
            test_name, verdict = item.split(":", 1)
            normalized.append({"test": test_name.strip(), "result": verdict.strip()})
    elif isinstance(test_results, dict):
        for test_name, verdict in test_results.items():
            normalized.append({"test": str(test_name).strip(), "result": str(verdict).strip()})
    return normalized


def add_observation(obs_list, key, level, message, evidence):
    obs_list.append({
        "key": key,
        "level": level,
        "message": message,
        "evidence": evidence,
    })

## Cell 6 Guide: Build LLM-Ready Context From ROBERT Outputs

This cell only captures ROBERT-generated evidence from `run_context.json` and organizes it for downstream chat prompts.

It does not compute a new score, does not invent score components, and does not introduce new warning thresholds.

Any convenience math is labeled as derived and never presented as a ROBERT score substitute.

In [31]:
pred_type = context.get("pred_type")

# This artifact is designed as LLM-ready context built from ROBERT outputs.
# It must not introduce a competing score system.
diag = {
    "schema_version": "2.0",
    "diagnosed_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "results_dir": context.get("results_dir"),
    "pred_type": pred_type,
    "ml_model": context.get("ml_model"),
    "dataset_csv": context.get("dataset_csv"),
    "robert_command": context.get("robert_command"),
    "available": context.get("available", {}),
    "robert_score": context.get("score", {}),
    "evidence": {
        "predict": context.get("predict", {}),
        "verify": context.get("verify", {}),
        "curate": context.get("curate", {}),
        "parser_warnings": context.get("parser_warnings", []),
    },
    "observations": {
        "no_pfi": [],
        "pfi": [],
        "global": [],
    },
    "llm_context": {
        "prompt_intent": (
            "Use ROBERT outputs as authority. Explain implications and next checks "
            "without inventing new score systems."
        ),
        "report_view_mode": "User may view PDF while LLM uses extracted ROBERT context.",
        "source_policy": {
            "pdf_to_llm": False,
            "raw_text_context": True,
            "image_context": True,
        },
        "citation_fields": [
            "predict.no_pfi",
            "predict.pfi",
            "verify.no_pfi",
            "verify.pfi",
            "curate",
            "score",
        ],
    },
    "derived_values": {
        "convenience_only": True,
        "items": {
            "no_pfi": {},
            "pfi": {},
        },
    },
    "notes": [],
}

for variant in ["no_pfi", "pfi"]:
    p = context.get("predict", {}).get(variant, {})
    v = context.get("verify", {}).get(variant, {})
    obs = diag["observations"][variant]

    # VERIFY outcomes as ROBERT reported them
    verify_items = normalize_verify_results(v.get("test_results"))
    if verify_items:
        for item in verify_items:
            add_observation(
                obs,
                key=f"verify_{item['test']}",
                level=item["result"],
                message=f"ROBERT VERIFY: {item['test']} -> {item['result']}",
                evidence={
                    "source": f"verify.{variant}.test_results",
                    "test": item["test"],
                    "result": item["result"],
                },
            )
    else:
        add_observation(
            obs,
            key="verify_results_missing",
            level="info",
            message="VERIFY test results were not present in extracted ROBERT outputs.",
            evidence={"source": f"verify.{variant}.test_results"},
        )

    # ROBERT-reported aggregate VERIFY counts
    for name in ["failed_tests", "unclear_tests", "passed_tests", "flawed_mod_score"]:
        if v.get(name) is not None:
            add_observation(
                obs,
                key=f"verify_{name}",
                level="info",
                message=f"ROBERT reported {name} = {v.get(name)}.",
                evidence={"source": f"verify.{variant}.{name}", "value": v.get(name)},
            )

    # Data shape observations from ROBERT outputs
    for name in ["n_train", "n_test", "n_descriptors", "points_descp_ratio", "train_outlier_pct", "test_outlier_pct"]:
        if p.get(name) is not None:
            add_observation(
                obs,
                key=f"predict_{name}",
                level="info",
                message=f"ROBERT reported {name} = {p.get(name)}.",
                evidence={"source": f"predict.{variant}.{name}", "value": p.get(name)},
            )

    # Convenience derived value: train:descriptor numeric ratio from ROBERT counts
    n_train, n_desc = parse_ratio(p.get("points_descp_ratio"))
    if n_train is not None and n_desc not in (None, 0):
        derived_ratio = n_train / n_desc
        diag["derived_values"]["items"][variant]["train_to_descriptor_ratio"] = {
            "value": round(derived_ratio, 3),
            "from": f"predict.{variant}.points_descp_ratio",
            "label": "derived convenience value",
        }

# Global notes for LLM usage and evidence gaps
if not context.get("score") or all(v is None for v in context.get("score", {}).values()):
    diag["notes"].append(
        "ROBERT score not yet populated in run_context.json for this run. "
        "Use ROBERT report outputs as authoritative source when available."
    )

if context.get("parser_warnings"):
    add_observation(
        diag["observations"]["global"],
        key="parser_warnings_present",
        level="info",
        message="Extractor reported parser warnings; cite evidence carefully.",
        evidence={"source": "parser_warnings", "value": context.get("parser_warnings")},
    )

print("Built ROBERT-grounded context package (no score recomputation).")
for variant in ["no_pfi", "pfi"]:
    print(f"- {variant}: {len(diag['observations'][variant])} observations")

Built ROBERT-grounded context package (no score recomputation).
- no_pfi: 11 observations
- pfi: 11 observations


## Cell 8 Guide: Write Output Artifacts

This cell writes `diagnosis.json` and `diagnosis_summary.md` next to `run_context.json`.

The summary mirrors ROBERT-reported evidence and observations; it does not present a separate agent score.

## Cell 7b Guide: Interpretation Pass — What Do These Results Mean?

This cell reads the already-extracted ROBERT evidence and adds meaning-level observations.

Each rule below asks: *given what ROBERT reported, what does that imply for this model?* The rules are deterministic — no LLM, no invented scores. All thresholds are labeled constants so a chemist can review and adjust them.

Observations added here go into the same structure as all other observations, with the key `interp_*` to distinguish them from direct echoes.

In [32]:
# -------------------------------------------------------------------------
# Interpretation thresholds (all labeled so a chemist can audit and adjust)
# -------------------------------------------------------------------------
# Minimum recommended molecules-per-descriptor for reliable cross-validation.
# Below this, CV variance is high and estimates should be treated with caution.
THRESHOLD_MIN_N_PER_DESCRIPTOR = 5

# Gap above which CV is considered meaningfully more optimistic than held-out test.
# Applies to both R2 (regression) and MCC (classification).
THRESHOLD_CV_TEST_GAP_WARNING = 0.15

# Gap above which test is meaningfully better than CV.
# Less common, but can indicate a favorable or unrepresentative split.
THRESHOLD_TEST_CV_GAP_INFO = 0.15

# Metric ceiling below which a model with all VERIFY tests PASSED is still considered
# 'consistent but not predictive'. Models below this threshold may have the right
# internal consistency but descriptors that don't correlate with the target property.
THRESHOLD_WEAK_MODEL_R2 = 0.5   # for regression
THRESHOLD_WEAK_MODEL_MCC = 0.5  # for classification

# Feature importance: if the top descriptor accounts for more than this fraction
# of total permutation importance, it dominates the model.
THRESHOLD_DOMINANT_FEATURE = 0.7


def get_primary_metric(p, pred_type):
    """Return (cv_metric, test_metric, metric_name) for the run type."""
    if pred_type == 'reg':
        return p.get('r2_cv'), p.get('r2_test'), 'R2'
    return p.get('mcc_cv') or p.get('r2_cv'), p.get('mcc_test') or p.get('r2_test'), 'MCC'


def all_verify_passed(v):
    """Return True if every available VERIFY test PASSED (none FAILED or UNCLEAR)."""
    items = v.get('test_results') or []
    return len(items) > 0 and all('PASSED' in r for r in items)


for variant in ['no_pfi', 'pfi']:
    p = context.get('predict', {}).get(variant, {})
    v = context.get('verify', {}).get(variant, {})
    obs = diag['observations'][variant]

    cv_metric, test_metric, metric_name = get_primary_metric(p, pred_type)
    n_train_val, n_desc_val = parse_ratio(p.get('points_descp_ratio'))

    # --- Rule 1: Dataset too small for number of descriptors ---
    if n_train_val is not None and n_desc_val not in (None, 0):
        ratio_val = n_train_val / n_desc_val
        if ratio_val < THRESHOLD_MIN_N_PER_DESCRIPTOR:
            add_observation(
                obs,
                key='interp_low_n_per_descriptor',
                level='warning',
                message=(
                    f'Only {n_train_val} training molecules for {n_desc_val} descriptors '
                    f'(ratio {ratio_val:.1f}:1). Cross-validation estimates have high variance '
                    f'at this ratio. Model performance may look better or worse than it truly is, '
                    f'and generalisation to new chemical space is uncertain.'
                ),
                evidence={
                    'source': f'predict.{variant}.points_descp_ratio',
                    'n_train': n_train_val, 'n_descriptors': n_desc_val,
                    'ratio': round(ratio_val, 2),
                    'threshold': THRESHOLD_MIN_N_PER_DESCRIPTOR,
                },
            )

    # --- Rule 2: All VERIFY passed, but model is weak ---
    if (cv_metric is not None and all_verify_passed(v)):
        threshold = THRESHOLD_WEAK_MODEL_MCC if pred_type == 'clas' else THRESHOLD_WEAK_MODEL_R2
        if cv_metric < threshold:
            add_observation(
                obs,
                key='interp_consistent_but_weak',
                level='warning',
                message=(
                    f'All VERIFY tests PASSED, but {metric_name} CV = {cv_metric:.2f} is below {threshold}. '
                    f'This usually means the model is internally consistent (not fitting noise), '
                    f'but the chosen descriptors do not capture the chemistry that drives the target property. '
                    f'Consider whether the descriptors relate to the physical mechanism of your reaction or property.'
                ),
                evidence={
                    'source': f'predict.{variant}.{"mcc_cv" if pred_type == "clas" else "r2_cv"}',
                    'cv_metric': cv_metric, 'metric_name': metric_name,
                    'threshold': threshold,
                },
            )

    # --- Rule 3: CV was more optimistic than test (possible mismatch) ---
    if cv_metric is not None and test_metric is not None:
        gap = cv_metric - test_metric
        if gap > THRESHOLD_CV_TEST_GAP_WARNING:
            add_observation(
                obs,
                key='interp_cv_optimistic',
                level='warning',
                message=(
                    f'{metric_name} CV = {cv_metric:.2f}, Test = {test_metric:.2f} (gap = {gap:.2f}). '
                    f'CV performance was more optimistic than the held-out test. '
                    f'This can happen when training and test molecules come from different chemical regions, '
                    f'or when the dataset is too small for stable splits.'
                ),
                evidence={
                    'source': f'predict.{variant}',
                    'cv_metric': cv_metric, 'test_metric': test_metric,
                    'gap': round(gap, 3), 'threshold': THRESHOLD_CV_TEST_GAP_WARNING,
                },
            )

        # --- Rule 4: Test better than CV (unusual, note it) ---
        elif (test_metric - cv_metric) > THRESHOLD_TEST_CV_GAP_INFO:
            add_observation(
                obs,
                key='interp_test_better_than_cv',
                level='info',
                message=(
                    f'{metric_name} Test = {test_metric:.2f} is notably higher than CV = {cv_metric:.2f}. '
                    f'For a small dataset, this may reflect a favorable test split rather than true performance. '
                    f'Interpret with caution if the test set has fewer than ~10 molecules.'
                ),
                evidence={
                    'source': f'predict.{variant}',
                    'cv_metric': cv_metric, 'test_metric': test_metric,
                    'gap': round(test_metric - cv_metric, 3),
                },
            )

    # --- Rule 5: Single descriptor dominates feature importance ---
    fi = p.get('feature_importance_summary') or []
    if fi:
        total_inf = sum(f.get('influence_rmse', 0) or 0 for f in fi)
        if total_inf > 0:
            top = max(fi, key=lambda f: f.get('influence_rmse', 0) or 0)
            top_frac = (top.get('influence_rmse', 0) or 0) / total_inf
            if top_frac > THRESHOLD_DOMINANT_FEATURE:
                add_observation(
                    obs,
                    key='interp_dominant_feature',
                    level='info',
                    message=(
                        f'Descriptor "{top["descriptor"]}" accounts for {top_frac*100:.0f}% of feature importance. '
                        f'This may mean the other descriptors are redundant, or that the model relies '
                        f'heavily on a single chemical property. Consider whether this descriptor '
                        f'captures the core mechanism of your reaction or property.'
                    ),
                    evidence={
                        'source': f'predict.{variant}.feature_importance_summary',
                        'top_descriptor': top['descriptor'],
                        'influence_fraction': round(top_frac, 3),
                        'threshold': THRESHOLD_DOMINANT_FEATURE,
                    },
                )

    # --- Rule 6: ROBERT-reported correlation warning ---
    model_warnings = p.get('warnings') or []
    for w in model_warnings:
        if 'correlation' in str(w).lower() or 'corr' in str(w).lower():
            add_observation(
                obs,
                key='interp_descriptor_correlation',
                level='warning',
                message=(
                    f'ROBERT warned: {w} '
                    f'Correlated descriptors can inflate apparent model performance in cross-validation, '
                    f'because the model can reconstruct a removed descriptor from correlated ones. '
                    f'The PFI variant (if available) uses only the most important descriptor, '
                    f'which avoids this issue.'
                ),
                evidence={
                    'source': f'predict.{variant}.warnings',
                    'warning': str(w),
                },
            )


# --- Global: build a plain-language summary from interpretation observations ---
interp_messages = []
for variant in ['no_pfi', 'pfi']:
    for ob in diag['observations'][variant]:
        if ob['key'].startswith('interp_') and ob['level'] in ('warning', 'failed'):
            interp_messages.append(ob['message'])

if interp_messages:
    summary_text = ' '.join(interp_messages[:3])  # top 3 for brevity
    add_observation(
        diag['observations']['global'],
        key='interp_summary',
        level='warning',
        message=summary_text,
        evidence={'source': 'interpretation_pass', 'observation_count': len(interp_messages)},
    )
    print(f'Interpretation pass: {len(interp_messages)} interpretation observations generated.')
else:
    add_observation(
        diag['observations']['global'],
        key='interp_summary',
        level='info',
        message='No notable interpretation flags were raised. Evidence is consistent with a well-behaved model.',
        evidence={'source': 'interpretation_pass', 'observation_count': 0},
    )
    print('Interpretation pass: no notable flags.')

Interpretation pass: 4 interpretation observations generated.


In [33]:
diagnosis_path = RUN_FOLDER / "diagnosis.json"
summary_path = RUN_FOLDER / "diagnosis_summary.md"

diagnosis_path.write_text(json.dumps(diag, indent=2), encoding="utf-8")

lines = []
lines.append("# ROBERT Diagnosis Summary")
lines.append("")
lines.append(f"- Diagnosed at: {diag['diagnosed_at']}")
lines.append(f"- Prediction type: {diag.get('pred_type')}")
lines.append(f"- Model: {diag.get('ml_model')}")
lines.append(f"- Dataset: {diag.get('dataset_csv')}")
lines.append("")
lines.append("## Operating Mode")
lines.append("- ROBERT is authoritative for model scores and metrics.")
lines.append("- This artifact is extracted context for explanation, not a competing score.")
lines.append("- PDF is for user display; this extracted context is the ground truth for the UI.")

score_block = diag.get("robert_score", {})
lines.append("")
lines.append("## ROBERT Score (As Extracted)")
if score_block and any(v is not None for v in score_block.values()):
    for key in ["no_pfi", "pfi"]:
        lines.append(f"- {key}: {score_block.get(key)}")
else:
    lines.append("- Not available in this run_context.json (null placeholders).")

# --- Interpretation summary (plain-language flags) ---
global_interp = [ob for ob in diag['observations'].get('global', []) if ob['key'] == 'interp_summary']
if global_interp:
    lines.append("")
    lines.append("## Interpretation (What This Means)")
    for ob in global_interp:
        lines.append(f"{ob['message']}")

for variant in ["no_pfi", "pfi"]:
    p = diag["evidence"].get("predict", {}).get(variant, {})
    v = diag["evidence"].get("verify", {}).get(variant, {})
    obs = diag["observations"].get(variant, [])

    lines.append("")
    lines.append(f"## {variant.upper()} (ROBERT-Reported Evidence)")

    if pred_type == "reg":
        lines.append(f"- R2 CV/Test: {p.get('r2_cv')} / {p.get('r2_test')}")
        lines.append(f"- RMSE CV/Test: {p.get('rmse_cv')} / {p.get('rmse_test')}")
        lines.append(f"- MAE CV/Test: {p.get('mae_cv')} / {p.get('mae_test')}")
    else:
        lines.append(f"- MCC CV/Test: {p.get('mcc_cv')} / {p.get('mcc_test')}")

    lines.append(f"- Train/Test points: {p.get('n_train')} / {p.get('n_test')}")
    lines.append(f"- Descriptors: {p.get('n_descriptors')} ({p.get('descriptors')})")
    lines.append(f"- Train:descriptor string: {p.get('points_descp_ratio')}")

    lines.append("")
    lines.append("### VERIFY")
    lines.append(f"- failed/unclear/passed: {v.get('failed_tests')} / {v.get('unclear_tests')} / {v.get('passed_tests')}")
    lines.append(f"- flawed_mod_score (ROBERT): {v.get('flawed_mod_score')}")
    lines.append(f"- test_results: {v.get('test_results')}")

    # --- Interpretation observations for this variant ---
    interp_obs = [ob for ob in obs if ob['key'].startswith('interp_')]
    if interp_obs:
        lines.append("")
        lines.append("### What This Means")
        for item in interp_obs:
            lines.append(f"- [{item['level'].upper()}] {item['message']}")

    lines.append("")
    lines.append("### All Observations")
    if not obs:
        lines.append("- No observations recorded.")
    for item in obs:
        lines.append(f"- [{item['level']}] {item['key']}: {item['message']}")

lines.append("")
lines.append("## Derived Convenience Values")
lines.append("- These are convenience transforms from ROBERT-reported fields and are not ROBERT score components.")
for variant in ["no_pfi", "pfi"]:
    derived = diag.get("derived_values", {}).get("items", {}).get(variant, {})
    lines.append(f"- {variant}: {derived}")

if diag.get("notes"):
    lines.append("")
    lines.append("## Notes")
    for note in diag["notes"]:
        lines.append(f"- {note}")

summary_path.write_text("\n".join(lines), encoding="utf-8")

print(f"Wrote: {diagnosis_path}")
print(f"Wrote: {summary_path}")
print()
print("Summary preview:")
print("\n".join(lines[:50]))

Wrote: /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260516_150946__TOF_class/diagnosis.json
Wrote: /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive/20260516_150946__TOF_class/diagnosis_summary.md

Summary preview:
# ROBERT Diagnosis Summary

- Diagnosed at: 2026-05-17T01:08:37Z
- Prediction type: clas
- Model: RF
- Dataset: /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Clasification/AQME-ROBERT_interpret_TOF_clasif.csv

## Operating Mode
- ROBERT is authoritative for model scores and metrics.
- This artifact is extracted context for explanation, not a competing score.
- PDF is for user display; this extracted context is the ground truth for the UI.

## ROBERT Score (As Extracted)
- Not available in this run_context.json (null placeholders).

## Interpretation (What This Means)
Only 14.0 training molecules for 6.0 descriptors (ratio 2.3:1). Cross-validation estimates have high variance at this ratio. Model performance may look better or worse than it truly is

## Usage Notes

1. Run `agent/extract_context.ipynb` first to generate `run_context.json` from ROBERT outputs.
2. In Cell 3, set `RUN_CONTEXT_PATH` if you want a specific run; otherwise keep `None`.
3. Run all cells in order.
4. Inspect `diagnosis.json` and `diagnosis_summary.md` in the selected run folder.
5. This notebook builds LLM-ready context from ROBERT outputs; it does not compute an alternative score.
6. The user can read the PDF in the UI while the LLM uses extracted raw context and image references behind the scenes.